## GPT prompting

### requires python >= 3.10

In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import numpy as np
import copy
import tiktoken
import openai 
from gpt_cost_estimator import CostEstimator
import os
from openai import AzureOpenAI
import configparser
import json
import time
import pydantic
from pydantic import Field
from typing import Literal, List, Dict
from enum import Enum
from pydantic import BaseModel
import outlines
from outlines.models.openai import OpenAI, OpenAIConfig
from pydantic import ValidationError
import csv

In [2]:
pd.set_option('display.max_colwidth', None)

# OSA I : Andmed

## V1: teha kui pole olemas csv faili andmetega, muidu V2

### see teeb korrektse uue andmefaili v2

### andmetabelid

In [3]:
filename = "../drive_data/v33_koondkorpus_transaktsioonid_v04_2.db"
conn = sqlite3.connect(filename)
cursor = conn.cursor()

### graafiku punktide info

In [4]:
query = """SELECT verb, verb_compound, morph_case, log2_ratio, level,unique_lemmas, ann_unique_lemmas, 
            not_ann_unique_lemmas, olulisus, my_tag, other_tags, annotated, not_annotated, verb_case_count
            FROM lines_class_info4
            """

class_info = pd.read_sql(query, conn)
class_info

,verb,verb_compound,morph_case,log2_ratio,level,unique_lemmas,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated,verb_case_count
0,aasima,,ad,-9.965784,-,1,NaN,5.0,-,0,1,1,7,8
1,abistama,,all,-9.965784,-,1,NaN,12.0,-,0,1,1,15,16
2,aeglustama,,all,-9.965784,-,1,NaN,7.0,-,0,1,1,8,9
3,aerutama,,in,-9.965784,-,1,NaN,5.0,-,0,3,3,10,13
4,aevastama,,in,9.965784,-,1,1.0,6.0,-,1,0,1,6,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20935,õnnestuma,,ad,-1.971847,-,1313,331.0,2522.0,-,1022,4009,5031,17243,22274
20936,õppima,,in,5.233872,n90,530,463.0,985.0,0.0,5720,152,5872,5891,11763
20937,ütlema,,ad,-5.370614,n10,415,104.0,1162.0,0.0,295,12205,12500,9966,22466
20938,ütlema,,all,0.271387,n70,1131,189.0,2337.0,1.0,9354,7750,17104,40742,57846


### võtta ainult n80 tsooni lõksud

In [5]:
filtered_class = class_info[class_info["level"]=="n80"]
filtered_class = filtered_class.sort_values(["olulisus"])

In [6]:
filtered_class['olulisus'] = filtered_class['olulisus'].astype(float)

In [7]:
filtered_class

,verb,verb_compound,morph_case,log2_ratio,level,unique_lemmas,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated,verb_case_count
19889,peatuma,,in,3.551257,n80,282,257.0,337.0,0.00000,973,83,1056,858,1914
19994,möllama,,in,3.921070,n80,196,177.0,261.0,0.00000,409,27,436,495,931
20637,leiduma,,in,3.086569,n80,518,415.0,1689.0,0.00000,1614,190,1804,4760,6564
20673,naasma,,el,3.210249,n80,287,236.0,222.0,0.00000,907,98,1005,439,1444
19859,süttima,,in,3.879146,n80,199,177.0,233.0,0.00000,515,35,550,661,1211
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18469,nappima,,in,3.496426,n80,140,114.0,198.0,0.00006,316,28,344,382,726
18887,sõitma,edasi,el,5.741467,n80,55,53.0,26.0,0.00007,107,2,109,30,139
20015,teatama,,ill,5.235216,n80,54,51.0,82.0,0.00008,113,3,116,640,756
20194,ootama,,ill,3.882643,n80,108,93.0,132.0,0.00008,236,16,252,231,483


In [8]:
filtered_class.to_sql("lines_class_info4_n80", conn, if_exists="replace", index=False)

335

### võtta spatial_obl tabelist näitelaused koos vajaliku infoga

In [27]:
# spatial obl_tabelist lõksud, mis on lines_class_info4_n80 tabelis
# iga lõksu kohta max 500 lemmat ja iga unikaalse lemma kohta 1 näide


query = """
WITH cleaned AS (
    -- Step 1 & 2: match subset table + remove rows with timex_tag NOT NULL
    SELECT
        d.head_id,
        d.form,
        d.lemma,
        d.verb,
        d.verb_compound,
        d.morph_case,
        d.sentence,
        d.sentence_id,
        d.timex_tag,
        d.ekilex_tag,
        d.ner_tag
    FROM spatial_obl AS d
    JOIN lines_class_info4_n80 AS s
      ON d.verb = s.verb
     AND d.verb_compound = s.verb_compound
     AND d.morph_case = s.morph_case
    WHERE d.timex_tag IS NULL
),

distinct_lemmas AS (
    -- Step 3 & 4: for each lemma, pick ONE sentence deterministically
    SELECT 
        *,
        ROW_NUMBER() OVER (
            PARTITION BY verb, verb_compound, morph_case, lemma
            ORDER BY sentence_id   -- choose best or earliest sentence
        ) AS rn_per_lemma
    FROM cleaned
),

limited AS (
    -- Step 5: limit to 500 unique lemmas per (verb, verb_compound, morph_case)
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY verb, verb_compound, morph_case
            ORDER BY lemma        -- choose 500 lexicographically smallest lemmas
        ) AS rn_group_limit
    FROM distinct_lemmas
    WHERE rn_per_lemma = 1      -- keep only one sentence per lemma
)

-- Step 6: final output
SELECT
    head_id,
    form,
    lemma,
    verb,
    verb_compound,
    morph_case,
    sentence,
    sentence_id,
    timex_tag,
    ekilex_tag,
    ner_tag
FROM limited
WHERE rn_group_limit <= 500
ORDER BY verb, verb_compound, morph_case, lemma;

"""


spatial_obl_ex = pd.read_sql(query, conn)


In [28]:
spatial_obl_ex

,head_id,form,lemma,verb,verb_compound,morph_case,sentence,sentence_id,timex_tag,ekilex_tag,ner_tag
0,23747396,17das,17.,ajama,,in,"Mingi aeg ajasin täna 17das servus kõiki Great Forge'i hüppama , isegi sain mõned",15208349,None,None,None
1,18645542,Aafrikas,Aafrika,ajama,,in,"Kas teadsite , et Brechti suu oli nagu mustade hauakividega surnuaed , et Hemingway ajas Aafrikas pea kiilaks ja lõhkus neegritüdrukuga välivoodi , et Bertrand Russell polnud võimeline endale teed keetma isegi siis , kui tal oli käes kirjalik juhend , kuidas seda teha .",11648113,None,location,LOC
2,26620253,Afganistanis,Afganistan,ajama,,in,"Afganistanis ja Iraagis musklid suureks ajanud Bush on olnud valmis kohe-kohe karistama « kurjuse telje » põhiliiget Iraani , ent mida kehvemalt on käinud käbarad kahel sõjatandril , seda vähemmõjuvaks on muutunud ka Bushi jutt Iraanist , mida nüüd , ennäe , väisas USAs ja Euroopas igast uksest ja aknast sisse lastud sõber Putin ise .",17346120,None,location,LOC
3,25724171,Alžeerias,Alžeeria,ajama,,in,"Alžeerias ajas mind nii naerma - ümberringi ainult liiv , liiv , liiv ...",16719535,None,location,LOC
4,17515699,Ameerikas,Ameerika,ajama,,in,"Lõppude lõpuks , kui Ameerikas ajavad metalpopi liini Smashing Pumpkins ja Foo Fighters , siis oleme meie siin Blindi väärt küll .",10936172,None,location,LOC
...,...,...,...,...,...,...,...,...,...,...,...
75407,21960802,väikelinnas,väikelinn,üürima,,in,Arnold üürib väikelinnas korterit .,13774030,None,location,None
75408,6229577,võõrastemajas,võõrastemaja,üürima,,in,"Haritud keskealine mees meenutab üht oma kunagist armulugu , mis algab sellega , et ta reisib välismaale ja üürib võõrastemajas toa .",3867933,None,location,None
75409,9515834,äärelinnas,äärelinn,üürima,,in,Noormees üürib väikese korteri äärelinnas .,5927429,None,location,None
75410,2889349,ühiselamus,ühiselamu,üürima,,in,"Üliõpilane Viljar ( 22 ) üürib tuba Mustamäel Vilde teel poollagunenud ühiselamus , sest peab kalli korteri üürimist ebaotstarbekaks ning isikliku elamispinna ostuks pole raha .",1815434,None,location,None


In [30]:
# shuffle
df = spatial_obl_ex.sample(frac=1)

In [31]:
df.to_csv("../gpt_input/n80_examples_large_v2.csv", encoding="utf-8", index = False, sep="|")

In [32]:
spatial_obl_ex.to_csv("../gpt_input/n80_examples_large_v2_sorted.csv", encoding="utf-8", index = False, sep="|")

In [3]:
df2 = pd.read_csv("../gpt_input/n80_examples_large_v2.csv", encoding="utf-8", sep="|")

In [4]:
counts2 = df2.groupby(['verb','verb_compound', 'morph_case'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)
counts2

,verb,verb_compound,morph_case,count
332,õpetama,NaN,in,500
0,ajama,NaN,in,500
3,andma,NaN,ill,500
274,tegelema,NaN,in,500
285,tooma,NaN,adit,500
...,...,...,...,...
272,tarnima,NaN,ill,37
296,turustama,NaN,in,35
121,lendama,edasi,ill,35
241,soetama,NaN,ill,34


In [33]:
conn.close()

## V2 andmed: kui on eelnevalt salvestatud csv siis lugeda sisse

## testimise põhjusel on kasutusel vana v1 andmefail, et tulemusi saaks võrrelda

In [3]:
#df = pd.read_csv("../gpt_input/n80_examples_large_v1.csv", encoding="utf-8",  sep="|")
df = pd.read_csv("../gpt_output/n80_examples_large_v1_gpt_v1_10K_b12_v1.csv", encoding="utf-8",  sep="|")

In [4]:
len(df)

10000

In [5]:
# kui faili on laused salvestatud shufflitud olekus, siis võiks võtta lihtsalt esimesed n

spatial_obl_ex = df.iloc[:10000]
spatial_obl_ex = spatial_obl_ex.sample(frac=1)

In [6]:
spatial_obl_ex

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
1551,4477403,Tallinnas,Tallinn,sadama,maha,in,2792027,Tallinnas laupäeval maha sadanud lumi lõi ilmajaama andmeil kümne aasta rekordi .,NaN,location,LOC,yes,NaN
2378,3433963,piletiäris,piletiäri,ringlema,NaN,in,2149636,"Lihtsad arvutused näitavad , et Tallinna põrandaaluses piletiäris ringlevad summad on tohutud .",NaN,NaN,NaN,no,"The term 'piletiäris' refers to ticket trading and is not a location, so it was classified as 'no'."
515,19808269,fuajees,fuajee,sööma,NaN,in,12372585,"Etenduse vaheajal sõid lapsed teatri fuajees puuvilju ning mängis ansambel "" Üks lust "" .",NaN,location,NaN,yes,NaN
4192,10334047,Õnnetuspaika,õnnetuspaik,kiirustama,NaN,adit,6427967,Õnnetuspaika kiirustanud Soome ja Eesti päästekopterid meest enne pimeduse saabumist ei leidnud .,NaN,location,NaN,yes,NaN
5390,21705880,Vilniusesse,Vilnius,lubama,NaN,ill,13574774,"SK Polaris ei lubanud Vilniusesse Jaanus Liivakut , nii tugevdavad Kalevit Valmo Kriisa Nybitist ja esmakordselt Kristo Reinumäe Canon-Eesti noortemeeskonnast .",NaN,location,LOC,yes,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2497,9245685,teokarbist,teokarp,voolama,välja,el,5763409,"Ka vetejumala jalgade juures olevast teokarbist voolab välja vesi , mis valgub mööda kaskaadi astmeid allapoole .",NaN,NaN,NaN,no,"The phrase 'teokarbist' refers to an object (a seashell) rather than a geographical location, so it was classified as 'no'."
8257,25865186,Thbilisis,Thbilisi,varisema,kokku,in,16806790,Thbilisis varises kokku kaks elamut .,NaN,location,LOC,yes,"The word 'Thbilisis' refers to the city of Tbilisi, a specific geographic location, so it is classified as 'yes'."
9374,12338914,nimekirjadesse,nimekiri,laskma,NaN,ill,7695875,"Ilma arstiabita ei jää ka need , kes ennast nimekirjadesse ei lase kanda , kinnitab Hillar Kalda .",NaN,NaN,NaN,no,"The phrase 'nimekirjadesse' refers to lists, which are not a location, so it was classified as 'no'."
3549,10056376,linnusesse,linnus,toimuma,NaN,ill,6260170,"20. augusti õhtul toimub rongkäik Rakvere spordihallist linnusesse , kus kella 23ni toimub rahvapidu .",NaN,location,NaN,yes,NaN


# OSA II : GPT

## GPT jaoks vajalik

In [7]:
config = configparser.ConfigParser()

conf_file = '../azure.ini'

status = config.read(conf_file) 
assert status == [conf_file]

API_VERSION = config['azure-configuration']['api_version']
AZURE_ENDPOINT = config['azure-configuration']['api_base']
SUBSCRIPTION_KEY = config['azure-configuration']['api_key']
model_name = "gpt-4o" #"GPT-4o-2024-1120 Global"
DEPLOYMENT = config['azure-configuration']['deployment_id']

In [8]:
def in2json(sisend):
    return json.dumps(sisend, ensure_ascii=False)


def user_message(lause, fraas):
    mes ={
            "role": "user",
            "content": {"l": lause, "c": fraas}
            }
    return mes


def assistant_message(yesno, short_ans, long_ans):
    mes = {
            "role": "assistant",
            "content": {"a": yesno, "s": short_ans, "r": long_ans}
            }
    return mes


def messages_2_str(messages:list[dict]):
    # kui tahta samad dict prompti asjad anda ette lihtsalt stringina
    # tekitab 5-realised blokid, eraldatud: \n\n
    
    new_string = ""
    
    for mes in messages:
        if mes["role"]=="user":
            l = mes["content"]["l"]
            c = mes["content"]["c"]
            new_string += f"l: {l}\n"
            new_string += f"c: {c}\n"
        if mes["role"]=="assistant":
            a = mes["content"]["a"]
            s = mes["content"]["s"]
            r = mes["content"]["r"]
            new_string += f"a: {a}\n"
            new_string += f"s: {s}\n"
            new_string += f"r: {r}\n\n"
    
    return new_string

In [9]:
SYSTEM_PROMPT = """
You are a classification assistant.
In this task location refers to "adverbial of place" (Estonian: kohamäärus) or "locative adverb".
Your task: Given a list of examples, each with keys "l" (sentence) and "c" (phrase), classify whether "c" functions as a location in the context of the sentence.
Adverbial of place answers to the question “where” (kus?/kuhu?/kust?) in the context of the sentence.
It is a place or concept where something or someone is located, goes to or comes from.
Criteria:
- concrete place (bank, table, Berlin)
- abstract (literature, soul, TV channels, government, top of a group, history, thought, domain)
- inanimate (journal, chair, wifi, bag, medal, toy, food, computer, wire, body parts)
- alive (mother, Peter, dog, doctor, teacher)
- event (dress rehearsal, camp, class, situation, meeting)
- state or condition conceptualized as space (life, trouble, consciousness, attitude)
- Locations ARE NOT phrases that show time, state of being, owner, experiencer, instrument, manner OR are purely grammatical constructions. 
- If the phrase can answer the question when, in what state, who, with what or how, then it is not a location.

Analyse the given criteria of location, analyse the 'few_shots' examples and generalise.
Then process the list called 'batch'.

Output JSON requirements:
- Respond strictly with an array of JSON objects, one object per 'batch' item.
- The JSON array must be in the exact same order as the batch items.
- Response must be without markdown or comments.
- Each output JSON must have:
  "a": "yes" (location) or "no" (not location)
"""

In [11]:
FEW_SHOTS = [
            user_message("Me läksime Pariisi.","Pariisi"),
            assistant_message("yes", "city", "City name, answers question 'where'."),
    
            user_message("Ema on mul olnud alati õmblustöö inimene ja õpetab seda praegu ühes õmbluskoolis teistelegi.", "õmbluskoolis"),
            assistant_message("yes", "location", "Organization's building, answers 'where'."),
    
            user_message("Põhjapoolusele saabus kottide-kompsudega tuhandeid võõrtöölisi.", "Põhjapoolusele"),
            assistant_message("yes", "location", "Answers the question 'to where'."),
    
            user_message("Ta tuli idast kõikide oma raamatutega.", "idast"),
            assistant_message("yes", "direction", "Abstract location, answers 'from where'."),
    
            user_message("Mees istus peale pikka päeva uuesti sadulasse.", "sadulasse"),
            assistant_message("yes", "object", "Man sat on an object and answers 'where'."),

            user_message("Nad sõidavad neljapäeval maale.", "neljapäeval"),
            assistant_message("no", "time", "Answers 'when'."),
    
            user_message("Avo süüdistati selles, et ta varastas magava J.P. põuetaskust raha koos rahakotiga.", "põuetaskust"),
            assistant_message("yes", "object", "Answers 'from where' and is an object."),
    
            user_message("Näiteks korraldas Affleck Lopezile üllatus-sünnipäevapeo restoranis Park.", "restoranis"),
            assistant_message("yes", "location", "Physical location and also organization, answers 'where'."),
    
            user_message("HP700 ei ulatu enam SpeedTouchi Wifi'sse.", "Wifi'sse"),
            assistant_message("yes", "abstract", "Abstract location, answers 'where'"),

            user_message("Minnie käis Barbra teadmata isegi kleidiproovis.", "kleidiproovis"),
            assistant_message("yes", "event", "Event that answers 'where' the person was."),

            user_message("Rüselejal käsisid sussid", "Rüselejal"),
            assistant_message("no", "experiencer", "Rüseleja is the experiencer."),
    
            user_message("Nüüd siis istun sitas.", "sitas"),
            assistant_message("no", "state", "State of being."),    
    
            user_message("Protest on mitmekesine ja teravaimalt avaldub see kirjanduses.", "teravaimalt"),
            assistant_message("no", "other", "Verbial of manner."),    
    
            user_message("Korraldasime seminari TTÜs.", "TTÜs"),
            assistant_message("yes", "location", "TTÜ is an organization but in sentence refers to location and answers 'where'."),
    
            user_message("Väga hästi varjab päikesekiiri näiteks markiis.", "markiis"),
            assistant_message("no", "other", "Nominative case and not location."),
    
            user_message("Ingridi puhul läks hiljem täkkesse just see ütelus.", "täkkesse"),
            assistant_message("no", "other", "Phrasal verb and not location."),
    
            user_message("Mari kuulas kikkis kõrvul.", "kõrvul"),
            assistant_message("no", "manner", "Answers the question 'how'."),
    
            user_message("Maril on kaks last.", "Maril"),
            assistant_message("no", "owner", "Answers the question 'who'."),
    
            user_message("See asi ununes mul täielikult.", "mul"),
            assistant_message("no", "experiencer", "Answers the question 'who'."),
    
            user_message("Luba tal ükskord ometi kõik südamelt ära rääkida.", "tal"),
            assistant_message("no", "experiencer", "Answers the question 'who'."),
    
            user_message("Tüdruku nägu on naerul.", "naerul"),
            assistant_message("no", "state", "Answers the question 'in what state'."),
    
            user_message("Munad on vahul.", "vahul"),
            assistant_message("no", "state", "Answers the question 'in what state'."),
    
            user_message("Mari elab juba kolmandat aastat välismaal.", "välismaal"),
            assistant_message("yes", "location", "Answers the question 'where'."),
    
            user_message("Üliõpilased on loengul.", "loengul"),
            assistant_message("yes", "location", "Answers the question 'where'."),
    
            user_message("Müts on peas.", "peas"),
            assistant_message("yes","location", "Answers the question ‘where’."),
    
            user_message("Jüri on Keskerakonnas.", "Keskerakonnas"),
            assistant_message("yes", "location", "Jüri is located in the organization's structure."),
    
            user_message("Kalle osaleb koalitsioonis.", "koalitsioonis"),
            assistant_message("yes", "location", "Kalle is located in the organization's structure."),
    
            user_message("Tema sünnipäev on märtsis.", "märtsis"),
            assistant_message("no", "time", "Answers question 'when'."),
    
            user_message("Mees on sügavas depressioonis.", "depressioonis"),
            assistant_message("no", "state", "Answers the question 'in what state'."),
    
            user_message("Ta on andekas matemaatikas.", "matemaatikas"),
            assistant_message("no", "construction", "Is purely grammatical."),
    
            user_message("Ma kahtlen teie siiruses.", "siiruses"),
            assistant_message("no", "construction", "Is purely grammatical."),
    
            user_message("Ta mängib orkestris.", "orkestris"),
            assistant_message("yes", "location", "Answers the question 'where'."),
    
            user_message("Rahvamurrus ei leidnud laps ema.", "rahvamurrus"),
            assistant_message("yes", "location", "Answers the question 'where'."),
    
            user_message("Me elame vabaduses, vendluses ja armastuses.", "vabaduses"),
            assistant_message("yes", "location", "Our existence is located in the concept of ‘vabadus’."),
    
            user_message("Sinus on midagi.", "sinus"),
            assistant_message("yes", "location", "Something like a feeling or potential can be located inside of ’sinus’."),
    
            user_message("Maxence märkas ema, hüppas diivanilt püsti ja lülitas televiisori välja, mis äratas emas kohe kahtlusi.", "emas"),
            assistant_message("yes", "location", "kahtlused are located inside of ema. Answers the question ‘where’"),
    
            user_message("Me kasvasime üles botastes.", "botastes"),
            assistant_message("no", "state", "Answers the question 'in what state'."),
    
            user_message("Moos valgus pirukast välja.", "pirukast"),
            assistant_message("yes", "location", "Answers the question 'from where'."),
    
            user_message("Nii voolab riiklikust meditsiinist elujõud muudkui välja .", "meditsiinist"),
            assistant_message("yes", "location", "Conspet, answers the question 'from where'."),
    
            user_message("Uuringu põhjal selgus , et ligi 70 protsenti naistest pöörduks pärast esimese lapse sündi heameelega vanasse töökohta tagasi.", "töökohta"),
            assistant_message("yes", "location", "Answers the question 'to where'."),
    
            user_message("Ema pani Peetrile teki peale.", "Peetrile"),
            assistant_message("yes", "location", "Peeter is not an experiencer in this context, answers the question 'to where'."),
    
            user_message("Toomas Lepp tegutses kaua ETV-s.", "ETV-s"),
            assistant_message("yes", "abstract", "Abstract location, answers 'where'."),
    
            user_message("Ta on lisanud Delfisse mitmeid artikleid.", "Delfisse"),
            assistant_message("yes", "abstract", "Abstract location, answers 'where'."),
    
            user_message("Ta istus hooaja jooksul peatreeneripingile.", "peatreeneripingile"),
            assistant_message("yes", "abstract", "Metaphorical physical movement, answers question 'to where'."),

            user_message("Elu läks rööbastesse tagasi.", "rööbastesse"),
            assistant_message("yes", "abstract", "Metaphorical physical movement, answers question 'to where'."),

            user_message("Organisatsiooni ladvikus on rahu.", "ladvikus"),
            assistant_message("yes", "location", "Peace is a state among the organization's top group, answers question 'where'."),

            user_message("Elus tuleb ikka takistusi ette.", "Elus"),
            assistant_message("yes", "location", "Life refers to abstract location, answers question 'where'."),

            user_message("Sakslastele valgus peale rünnakute laviin.", "Sakslastele"),
            assistant_message("yes", "location", "Germans are not experiences but abstract location of attacks, answers question 'onto where'."),

            user_message("Eile külastas pottseppa tema vana sõber.", "pottseppa"),
            assistant_message("no", "experiencer", "Answers 'who' was visited."),
             
            user_message("Ema sõidutab mind trenni.", "trenni"),
            assistant_message("yes", "location", "Answers the question 'where'."),
             
            user_message("Selles valguses paistavad asjad hullemad.", "valguses"),
            assistant_message("yes", "abstract", "Light becomes conceptual place, answers the question 'where'."),
             
            user_message("Ta viskas asjad kotti.", "kotti"),
            assistant_message("yes", "location", "Answers the question 'where'."),
             
            user_message("Mind saadeti uksest välja.", "uksest"),
            assistant_message("yes", "object", "Answers the question 'through where'."),
             
            user_message("Kurbus ei mahu näkku ära.", "näkku"),
            assistant_message("yes", "abstract", "Answers the question 'where'."),
             
            # kui lisada see, siis mudel arvab et "liblikas maandub emale" ei ole asukoht enam
            #user_message("Kui võtaks kellegi neist endaga kaasa?", "neist"),
            #assistant_message("no", "other", "Source set, not a spatial frame."),  
    
]


FEW_SHOTS_STR = messages_2_str(FEW_SHOTS)

In [12]:
FEW_SHOTS_STR

"l: Me läksime Pariisi.\nc: Pariisi\na: yes\ns: city\nr: City name, answers question 'where'.\n\nl: Ema on mul olnud alati õmblustöö inimene ja õpetab seda praegu ühes õmbluskoolis teistelegi.\nc: õmbluskoolis\na: yes\ns: location\nr: Organization's building, answers 'where'.\n\nl: Põhjapoolusele saabus kottide-kompsudega tuhandeid võõrtöölisi.\nc: Põhjapoolusele\na: yes\ns: location\nr: Answers the question 'to where'.\n\nl: Ta tuli idast kõikide oma raamatutega.\nc: idast\na: yes\ns: direction\nr: Abstract location, answers 'from where'.\n\nl: Mees istus peale pikka päeva uuesti sadulasse.\nc: sadulasse\na: yes\ns: object\nr: Man sat on an object and answers 'where'.\n\nl: Nad sõidavad neljapäeval maale.\nc: neljapäeval\na: no\ns: time\nr: Answers 'when'.\n\nl: Avo süüdistati selles, et ta varastas magava J.P. põuetaskust raha koos rahakotiga.\nc: põuetaskust\na: yes\ns: object\nr: Answers 'from where' and is an object.\n\nl: Näiteks korraldas Affleck Lopezile üllatus-sünnipäevapeo r

## pydantic

In [13]:
class ClassificationDict(BaseModel):
    a: Literal["yes", "no"]
    #results: List[Literal["yes", "no"]]

class ClassificationAnswer(BaseModel):
    form : dict

In [14]:
client = AzureOpenAI(
    api_version=API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=SUBSCRIPTION_KEY,
)

## Andmete söötmine

In [15]:
def classify_batch(my_batch):

    print("classify", len(my_batch))
    max_att = 1
    attempt = 0
    while attempt < max_att:
        attempt += 1
        user_payload = {
            "few_shots": FEW_SHOTS_STR,
            "batch": my_batch
        }
    
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": in2json(user_payload)}
        ]

        response = client.chat.completions.create(
            model=DEPLOYMENT,
            messages=messages,
            temperature=0, # absoluutselt min väljund 
        )

        raw_output = response.choices[0].message.content.strip()

        try:
            data = json.loads(raw_output)

            if len(data) != len(batch):
                raise ValueError(f"Väljundis ei ole õige arv vastuseid. Peaks olema {len(batch)} aga on {len(data)}.")
                
            elif len(data) == len(batch):
                for item in data:
                    ClassificationDict(**item)

            return response, raw_output

        except (ValidationError, json.JSONDecodeError, ValueError) as e:
            #print(f"Attempt {attempt} failed. Retrying batch...")
            print(f"Error: {e}")
            #print(f"Raw output: {raw_output[:500]}...")  # preview first 500 chars
            time.sleep(1)  # small delay before retry

    print(f"Batch failed after {max_att} attempts.")
    # isegi kui ei saanud kõike kätte siis saab pärast äkki käsitsi midagi juurde panna
    return response, raw_output


In [16]:
def explain_non_locations(
    batch: List[Dict[str, str]],
    yes_no_results: List[str],
    yes_subset_ratio: float = 0.0
) -> Dict[int, str]:

    # Determine which indices to explain
    no_indices = [i for i, r in enumerate(yes_no_results) if r["a"] == "no"]
    yes_indices = [i for i, r in enumerate(yes_no_results) if r["a"] == "yes"]

    # Diagnostic subset
    diag_count = int(len(yes_indices) * yes_subset_ratio)
    diag_indices = yes_indices[:diag_count]

    explain_indices = no_indices + diag_indices
    if not explain_indices:
        return None, None, None

    items_to_explain = [
        {
            "index": i,
            "l": json.loads(batch[i])["l"],
            "c": json.loads(batch[i])["c"],
            "classification": yes_no_results[i]
        }
        for i in explain_indices
    ]

    messages = [
        {"role": "system", 
         "content": ("Explain why each phrase 'c' was classified as adverbial of place ('yes') or not adverbial of place ('no') in sentence 'l'." 
                       "Give one sentence answer."
                        "You MUST return only a pure JSON object, without markdown and code fences. "
                        "The output must be a mapping: {index: explanation}. "
                        "Do not include ```json or any backticks. Do not include commentary.")
        },
        {"role": "user", "content": (
            """For EACH item without missing any, return a JSON object mapping index → explanation in this format '{"0": "explanation", "3": "explanation"}'.\n"""
            "Items:\n" + in2json(items_to_explain)
        )}
    ]

    response = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=messages,
    )

    raw = response.choices[0].message.content.strip()

    # ---- Pydantic validation ----
    try:
        ClassificationAnswer(form = json.loads(raw))
    except ValidationError as e:
        raise ValueError(f"Invalid JSON structure returned in explanations:\n{e}")

    return response, raw, explain_indices

In [17]:
#df = spatial_obl_ex.sample(frac=1)#.reset_index(drop=True)
df = spatial_obl_ex

In [18]:
df

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
1551,4477403,Tallinnas,Tallinn,sadama,maha,in,2792027,Tallinnas laupäeval maha sadanud lumi lõi ilmajaama andmeil kümne aasta rekordi .,NaN,location,LOC,yes,NaN
2378,3433963,piletiäris,piletiäri,ringlema,NaN,in,2149636,"Lihtsad arvutused näitavad , et Tallinna põrandaaluses piletiäris ringlevad summad on tohutud .",NaN,NaN,NaN,no,"The term 'piletiäris' refers to ticket trading and is not a location, so it was classified as 'no'."
515,19808269,fuajees,fuajee,sööma,NaN,in,12372585,"Etenduse vaheajal sõid lapsed teatri fuajees puuvilju ning mängis ansambel "" Üks lust "" .",NaN,location,NaN,yes,NaN
4192,10334047,Õnnetuspaika,õnnetuspaik,kiirustama,NaN,adit,6427967,Õnnetuspaika kiirustanud Soome ja Eesti päästekopterid meest enne pimeduse saabumist ei leidnud .,NaN,location,NaN,yes,NaN
5390,21705880,Vilniusesse,Vilnius,lubama,NaN,ill,13574774,"SK Polaris ei lubanud Vilniusesse Jaanus Liivakut , nii tugevdavad Kalevit Valmo Kriisa Nybitist ja esmakordselt Kristo Reinumäe Canon-Eesti noortemeeskonnast .",NaN,location,LOC,yes,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2497,9245685,teokarbist,teokarp,voolama,välja,el,5763409,"Ka vetejumala jalgade juures olevast teokarbist voolab välja vesi , mis valgub mööda kaskaadi astmeid allapoole .",NaN,NaN,NaN,no,"The phrase 'teokarbist' refers to an object (a seashell) rather than a geographical location, so it was classified as 'no'."
8257,25865186,Thbilisis,Thbilisi,varisema,kokku,in,16806790,Thbilisis varises kokku kaks elamut .,NaN,location,LOC,yes,"The word 'Thbilisis' refers to the city of Tbilisi, a specific geographic location, so it is classified as 'yes'."
9374,12338914,nimekirjadesse,nimekiri,laskma,NaN,ill,7695875,"Ilma arstiabita ei jää ka need , kes ennast nimekirjadesse ei lase kanda , kinnitab Hillar Kalda .",NaN,NaN,NaN,no,"The phrase 'nimekirjadesse' refers to lists, which are not a location, so it was classified as 'no'."
3549,10056376,linnusesse,linnus,toimuma,NaN,ill,6260170,"20. augusti õhtul toimub rongkäik Rakvere spordihallist linnusesse , kus kella 23ni toimub rahvapidu .",NaN,location,NaN,yes,NaN


In [19]:
def chunks(lst, size=10):
    """Yield successive chunks of size N."""
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

In [20]:
results = []
results2 = []
responses = []
explanations = []
explanations_all = {}

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10
# kui suure osa võtta "yes" vastustest "why" küsimusse
yes_subset_ratio = 0.2
batch_start_index = 0

batch_cnt = 0

rows = df.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in chunks(rows, size=bs):
    batch = []
    for ex in df_batch:
        batch.append(in2json({"l": ex["sentence"], "c": ex["form"]}))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    # võtab välja kõik batchis olnud "no" ja mõne "yes" ja küsib why
    expl_response, batch_explanations, answered_idx = explain_non_locations(
            batch=batch,
            yes_no_results=result_yesno,
            yes_subset_ratio=yes_subset_ratio
        )

    # mapping: vastused õige lause+fraasiga kokku
    if batch_explanations is not None:
        used_tokens += expl_response.usage.total_tokens
        
        explanations.append(json.loads(batch_explanations))
        
        # Map batch-local -> global indices
        for local_i, explanation in json.loads(batch_explanations).items():
            global_i = batch_start_index + int(local_i)
            explanations_all[global_i] = explanation

    batch_start_index += len(batch)

    if used_tokens >= 4500000:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    
    #break



classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
clas

classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
clas

In [32]:
used_tokens # batch 10-> 3700,  10K lauset -> ~ 7.9 eur

3726

In [21]:
len(results)

10000

In [22]:
faulty_batches = {}
faulty_answers = {}
num_full_batches = int(len(df)/bs)
partial_batches = False if num_full_batches*bs == len(df) else True

if len(results) == len(df):
    df["classification2"] = [r["a"] for r in results]

    new_explanations = []
    for i in range(len(df)):
        if i in explanations_all.keys():
            new_explanations.append( explanations_all[i])
        else:
            new_explanations.append("")
    
        
    df["explanation2"] = new_explanations 


else: # juhuks kui mudel ei anna õiget arvu vastuseid tagasi
    new_results = []
    new_explanations = []
    for b, (batchres, expl) in enumerate(zip(results2, explanations),start=0):
        expected_len = bs if b < num_full_batches else len(df)-(num_full_batches*bs)
        
        if len(batchres) != expected_len and b < num_full_batches: # pole poolikud batchid ja on puuduvaid vastuseid
            replacement = ["?" for i in range(expected_len)]
            #print(num_full_batches, b, len(replacement))
            new_results += replacement
            new_explanations += replacement
            faulty_batches[b] = batchres
            faulty_answers[b] = expl
        elif len(batchres) != expected_len and b >= num_full_batches and partial_batches:  # on poolikud batchid ja on puuduvaid vastuseid
            replacement = ["?" for i in range(len(expected_len))]
            new_results += replacement
            faulty_batches[b] = batchres
            new_explanations += replacement
            faulty_answers[b] = expl
        elif len(batchres) == expected_len: # kõik ok 
            new_results += [r["a"] for r in batchres]
            for i in range(len(batchres)):
                if str(i) in expl.keys():
                    new_explanations.append( expl[str(i)])
                else:
                    new_explanations.append("")
            
    
    df["classification2"] = new_results
    df["explanation2"] = new_explanations

    #df["explanation"] = new_explanations 

In [23]:
df

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,classification2,explanation2
1551,4477403,Tallinnas,Tallinn,sadama,maha,in,2792027,Tallinnas laupäeval maha sadanud lumi lõi ilmajaama andmeil kümne aasta rekordi .,NaN,location,LOC,yes,NaN,yes,"The phrase 'Tallinnas' specifies a location (in Tallinn), so it is adverbial of place."
2378,3433963,piletiäris,piletiäri,ringlema,NaN,in,2149636,"Lihtsad arvutused näitavad , et Tallinna põrandaaluses piletiäris ringlevad summad on tohutud .",NaN,NaN,NaN,no,"The term 'piletiäris' refers to ticket trading and is not a location, so it was classified as 'no'.",yes,
515,19808269,fuajees,fuajee,sööma,NaN,in,12372585,"Etenduse vaheajal sõid lapsed teatri fuajees puuvilju ning mängis ansambel "" Üks lust "" .",NaN,location,NaN,yes,NaN,yes,
4192,10334047,Õnnetuspaika,õnnetuspaik,kiirustama,NaN,adit,6427967,Õnnetuspaika kiirustanud Soome ja Eesti päästekopterid meest enne pimeduse saabumist ei leidnud .,NaN,location,NaN,yes,NaN,yes,
5390,21705880,Vilniusesse,Vilnius,lubama,NaN,ill,13574774,"SK Polaris ei lubanud Vilniusesse Jaanus Liivakut , nii tugevdavad Kalevit Valmo Kriisa Nybitist ja esmakordselt Kristo Reinumäe Canon-Eesti noortemeeskonnast .",NaN,location,LOC,yes,NaN,yes,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2497,9245685,teokarbist,teokarp,voolama,välja,el,5763409,"Ka vetejumala jalgade juures olevast teokarbist voolab välja vesi , mis valgub mööda kaskaadi astmeid allapoole .",NaN,NaN,NaN,no,"The phrase 'teokarbist' refers to an object (a seashell) rather than a geographical location, so it was classified as 'no'.",yes,
8257,25865186,Thbilisis,Thbilisi,varisema,kokku,in,16806790,Thbilisis varises kokku kaks elamut .,NaN,location,LOC,yes,"The word 'Thbilisis' refers to the city of Tbilisi, a specific geographic location, so it is classified as 'yes'.",yes,
9374,12338914,nimekirjadesse,nimekiri,laskma,NaN,ill,7695875,"Ilma arstiabita ei jää ka need , kes ennast nimekirjadesse ei lase kanda , kinnitab Hillar Kalda .",NaN,NaN,NaN,no,"The phrase 'nimekirjadesse' refers to lists, which are not a location, so it was classified as 'no'.",yes,
3549,10056376,linnusesse,linnus,toimuma,NaN,ill,6260170,"20. augusti õhtul toimub rongkäik Rakvere spordihallist linnusesse , kus kella 23ni toimub rahvapidu .",NaN,location,NaN,yes,NaN,yes,


In [32]:
df[(df["explanation2"]!="?") & (df["explanation2"]!="")]

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,classification2,explanation2
1551,4477403,Tallinnas,Tallinn,sadama,maha,in,2792027,Tallinnas laupäeval maha sadanud lumi lõi ilmajaama andmeil kümne aasta rekordi .,NaN,location,LOC,yes,NaN,yes,"The phrase 'Tallinnas' specifies a location (in Tallinn), so it is adverbial of place."
810,15008647,ekstaasis,ekstaas,tervitama,NaN,in,9344322,"Tahtsin lihtsalt lõpetada ja mõned punktid saada , ” tunnistas sakslane , keda ekstaasis publik autasustamispoodiumil ovatsioonide ja raketipaukudega tervitas .",NaN,NaN,NaN,no,"The word 'ekstaasis' translates to 'in ecstasy' and refers to an emotional state, not a physical location.",no,"The phrase 'ekstaasis' refers to an emotional state and does not indicate a physical location, so it is not adverbial of place."
7553,17784543,seisus,seis,ravima,NaN,in,11102573,"Kaks kuud ravis perenaine armetus seisus koera , kaotamata siiski lootust .",NaN,NaN,NaN,no,The phrase 'seisus' refers to 'condition' or 'state' and does not signify a location.,no,"The phrase 'seisus' refers to a condition or state rather than a location, so it is not adverbial of place."
818,18685172,House'is,House,liikuma,NaN,in,11673400,"House'is raha ei liigu , me teeme seda armastusest .",NaN,NaN,LOC,yes,"The term 'House'is' was classified as 'yes' because it refers to a specific named place, 'House'.",yes,The phrase 'House'is' is adverbial of place because it specifies the location where the action occurs (in the House).
9347,25172405,koosseisust,koosseis,minema,ära,el,16252383,"Kolm näidet , mis ma oskan öelda nende kohta , kes on ise lahkunud ( peale selle on lahkujaid ka seoses struktuuri ümberkorraldamisega , ministeeriumist on ära viidud näiteks haldusbüroo ja tehtud muid niisuguseid asju , mistõttu ministeeriumi koosseisust ära läinud isikuid on rohkem ) : näiteks Tallinna linn on saanud Haridusministeeriumist personalijuhi , kes oli töötanud ministeeriumis 13 aastat ja tahtis vaheldust ; üks daam läheb meil järgmisel nädalal Islandile mehele ; üks daam on läinud arvutiõpetajaks Rocca al Mare kooli .",NaN,NaN,NaN,no,The phrase 'koosseisust' pertains to organizational structure and not a geographic location.,no,"The phrase 'koosseisust' is not adverbial of place because it refers to membership or composition, rather than specifying a location."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4970,15120914,Zhigulisse,Zhiguli,istuma,NaN,ill,9417012,"Erariietes politseinik istus Zhigulisse , millel polnud politsei eraldusmärke .",NaN,NaN,LOC,yes,NaN,yes,"The phrase 'Zhigulisse' indicates the specific location where the policeman sat (a Zhiguli car), thus qualifying it as an adverbial of place."
2954,7534710,Vabariigis,Vabariik,laiuma,NaN,in,4689103,""" Palju loodusvõtteid tegime Põhja-Uuralites ja Euroopa suurimas rahvuspargis "" Jugõd Va "" ( Puhas Vesi ) , mis laiub Uuralitest läände voolavate jõgede vahel Komi Vabariigis Euroopa kirdenurgas .",NaN,NaN,LOC,yes,NaN,yes,"The phrase 'Vabariigis' indicates the location (state) where the action takes place, making it an adverbial of place."
6576,3912645,millest,mis,rändama,NaN,el,2440084,"17. jaanuaril avatakse Tallinna Kunstihoones John Smithi järjekordne suur ühisnäitus , millest märkimisväärne osa rändabki suve hakul Veneetsiasse .",NaN,NaN,NaN,no,"The phrase 'millest' does not refer to a specific geographic or physical location; instead, it relates to a part of the exhibition that will be moved.",no,"The phrase 'millest' refers to a part of the exhibition, not a physical location, so it is not an adverbial of place."
4689,5701350,käitudes,käit,kaevama,NaN,in,3540507,Nii käitudes kaevavad riigi tagant varastavad kalamehed Puuritsa arvates iseendale auku .,NaN,NaN,NaN,no,The word 'käitudes' refers to behavior or conduct and not a location.,no,"The phrase 'käitudes' describes the manner or action of behaving, not a location, so it is no

In [24]:
saving_fname = "../gpt_output/n80_examples_large_v1_gpt_v2_10K_b10_v1.csv"

In [25]:
df.to_csv(saving_fname, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)


In [26]:
fname1 = saving_fname[:-4] + "_faulty_batches.json"
with open(fname1, "w", encoding="utf-8") as f:
    json.dump(faulty_batches, f, ensure_ascii=False)

fname2 = saving_fname[:-4] + "_faulty_answers.json"
with open(fname2, "w", encoding="utf-8") as f:
    json.dump(faulty_answers, f, ensure_ascii=False)

fname1 = saving_fname[:-4] + "_yesno.json"
with open(fname1, "w", encoding="utf-8") as f:
    json.dump(results2, f, ensure_ascii=False)

fname2 = saving_fname[:-4] + "_why_reponses.json"
with open(fname2, "w", encoding="utf-8") as f:
    json.dump(explanations, f, ensure_ascii=False)

In [28]:
df[df["classification2"]=="yes"] # 8694

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,classification2,explanation2
1551,4477403,Tallinnas,Tallinn,sadama,maha,in,2792027,Tallinnas laupäeval maha sadanud lumi lõi ilmajaama andmeil kümne aasta rekordi .,NaN,location,LOC,yes,NaN,yes,"The phrase 'Tallinnas' specifies a location (in Tallinn), so it is adverbial of place."
2378,3433963,piletiäris,piletiäri,ringlema,NaN,in,2149636,"Lihtsad arvutused näitavad , et Tallinna põrandaaluses piletiäris ringlevad summad on tohutud .",NaN,NaN,NaN,no,"The term 'piletiäris' refers to ticket trading and is not a location, so it was classified as 'no'.",yes,
515,19808269,fuajees,fuajee,sööma,NaN,in,12372585,"Etenduse vaheajal sõid lapsed teatri fuajees puuvilju ning mängis ansambel "" Üks lust "" .",NaN,location,NaN,yes,NaN,yes,
4192,10334047,Õnnetuspaika,õnnetuspaik,kiirustama,NaN,adit,6427967,Õnnetuspaika kiirustanud Soome ja Eesti päästekopterid meest enne pimeduse saabumist ei leidnud .,NaN,location,NaN,yes,NaN,yes,
5390,21705880,Vilniusesse,Vilnius,lubama,NaN,ill,13574774,"SK Polaris ei lubanud Vilniusesse Jaanus Liivakut , nii tugevdavad Kalevit Valmo Kriisa Nybitist ja esmakordselt Kristo Reinumäe Canon-Eesti noortemeeskonnast .",NaN,location,LOC,yes,NaN,yes,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2497,9245685,teokarbist,teokarp,voolama,välja,el,5763409,"Ka vetejumala jalgade juures olevast teokarbist voolab välja vesi , mis valgub mööda kaskaadi astmeid allapoole .",NaN,NaN,NaN,no,"The phrase 'teokarbist' refers to an object (a seashell) rather than a geographical location, so it was classified as 'no'.",yes,
8257,25865186,Thbilisis,Thbilisi,varisema,kokku,in,16806790,Thbilisis varises kokku kaks elamut .,NaN,location,LOC,yes,"The word 'Thbilisis' refers to the city of Tbilisi, a specific geographic location, so it is classified as 'yes'.",yes,
9374,12338914,nimekirjadesse,nimekiri,laskma,NaN,ill,7695875,"Ilma arstiabita ei jää ka need , kes ennast nimekirjadesse ei lase kanda , kinnitab Hillar Kalda .",NaN,NaN,NaN,no,"The phrase 'nimekirjadesse' refers to lists, which are not a location, so it was classified as 'no'.",yes,
3549,10056376,linnusesse,linnus,toimuma,NaN,ill,6260170,"20. augusti õhtul toimub rongkäik Rakvere spordihallist linnusesse , kus kella 23ni toimub rahvapidu .",NaN,location,NaN,yes,NaN,yes,


In [30]:
df[(df["classification2"]=="yes") & (df["explanation2"]!="")]

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,classification2,explanation2
1551,4477403,Tallinnas,Tallinn,sadama,maha,in,2792027,Tallinnas laupäeval maha sadanud lumi lõi ilmajaama andmeil kümne aasta rekordi .,NaN,location,LOC,yes,NaN,yes,"The phrase 'Tallinnas' specifies a location (in Tallinn), so it is adverbial of place."
818,18685172,House'is,House,liikuma,NaN,in,11673400,"House'is raha ei liigu , me teeme seda armastusest .",NaN,NaN,LOC,yes,"The term 'House'is' was classified as 'yes' because it refers to a specific named place, 'House'.",yes,The phrase 'House'is' is adverbial of place because it specifies the location where the action occurs (in the House).
4926,2728007,Sadamast,sadam,astuma,läbi,el,1713888,"Või astume aasta algul Tallinna Sadamast läbi , et küsida kõiki rendilepinguid .",NaN,location,ORG,yes,NaN,yes,The phrase 'Sadamast' was classified as 'yes' because it indicates a location ('Tallinna Sadamast') and answers the question 'where'.
9908,24923713,eestisse,eestis,tooma,NaN,adit,16032430,"Kas keegi teab mõnda netipoodi mis tooks eestisse kaupu või teab mõnda poodi , mis müüb iPode odavamalt , kui minu otsitud hinnad ?",NaN,NaN,NaN,yes,NaN,yes,"The phrase 'eestisse' specifies a direction toward a location (Estonia), which qualifies it as adverbial of place."
394,917184,Suurbritannias,Suurbritannia,omama,NaN,in,579087,Näiteks Suurbritannias omab ISO sertifikaati iga seitsmes ettevõte .,NaN,location,LOC,yes,NaN,yes,"The phrase 'Suurbritannias' specifies the location where the action takes place, qualifying it as an adverbial of place."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6651,3945603,Mekasse,meka,lubama,NaN,ill,2461176,"Tervis lubaks muhameedlasi Mekasse küll , kuid rahakoti paksus jätab soovida .",NaN,location,LOC,yes,NaN,yes,"The phrase 'Mekasse' specifies the destination of the action (traveling), thus functioning as an adverbial of place."
1106,7474432,krais,krai,möllama,NaN,in,4651114,"Kõige hullemad tulekahjud olid Siberis , kus tuli möllas 1852 hektaril , suuremas jaos Krasnojarski krais .",NaN,NaN,NaN,yes,NaN,yes,"The phrase 'krais' specifies a particular location (Krasnojarski krai) where the fires were severe, making it an adverbial of place."
4970,15120914,Zhigulisse,Zhiguli,istuma,NaN,ill,9417012,"Erariietes politseinik istus Zhigulisse , millel polnud politsei eraldusmärke .",NaN,NaN,LOC,yes,NaN,yes,"The phrase 'Zhigulisse' indicates the specific location where the policeman sat (a Zhiguli car), thus qualifying it as an adverbial of place."
2954,7534710,Vabariigis,Vabariik,laiuma,NaN,in,4689103,""" Palju loodusvõtteid tegime Põhja-Uuralites ja Euroopa suurimas rahvuspargis "" Jugõd Va "" ( Puhas Vesi ) , mis laiub Uuralitest läände voolavate jõgede vahel Komi Vabariigis Euroopa kirdenurgas .",NaN,NaN,LOC,yes,NaN,yes,"The phrase 'Vabariigis' indicates the location (state) where the action takes place, making it an adverbial of place."


In [31]:
df[df["classification2"]=="no"] # 1306

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,classification2,explanation2
810,15008647,ekstaasis,ekstaas,tervitama,NaN,in,9344322,"Tahtsin lihtsalt lõpetada ja mõned punktid saada , ” tunnistas sakslane , keda ekstaasis publik autasustamispoodiumil ovatsioonide ja raketipaukudega tervitas .",NaN,NaN,NaN,no,"The word 'ekstaasis' translates to 'in ecstasy' and refers to an emotional state, not a physical location.",no,"The phrase 'ekstaasis' refers to an emotional state and does not indicate a physical location, so it is not adverbial of place."
7553,17784543,seisus,seis,ravima,NaN,in,11102573,"Kaks kuud ravis perenaine armetus seisus koera , kaotamata siiski lootust .",NaN,NaN,NaN,no,The phrase 'seisus' refers to 'condition' or 'state' and does not signify a location.,no,"The phrase 'seisus' refers to a condition or state rather than a location, so it is not adverbial of place."
9347,25172405,koosseisust,koosseis,minema,ära,el,16252383,"Kolm näidet , mis ma oskan öelda nende kohta , kes on ise lahkunud ( peale selle on lahkujaid ka seoses struktuuri ümberkorraldamisega , ministeeriumist on ära viidud näiteks haldusbüroo ja tehtud muid niisuguseid asju , mistõttu ministeeriumi koosseisust ära läinud isikuid on rohkem ) : näiteks Tallinna linn on saanud Haridusministeeriumist personalijuhi , kes oli töötanud ministeeriumis 13 aastat ja tahtis vaheldust ; üks daam läheb meil järgmisel nädalal Islandile mehele ; üks daam on läinud arvutiõpetajaks Rocca al Mare kooli .",NaN,NaN,NaN,no,The phrase 'koosseisust' pertains to organizational structure and not a geographic location.,no,"The phrase 'koosseisust' is not adverbial of place because it refers to membership or composition, rather than specifying a location."
7173,24205992,hooldeprojekti,hooldeprojekt,panema,NaN,adit,15500381,""" Vajaduse korral panevad Veerpalu ja loodetavasti õed Šmigunid isiklikku raha hooldeprojekti , sest kehvade suuskadega kaugele ei sõida .",NaN,NaN,NaN,no,"The phrase 'hooldeprojekti' refers to a project or initiative, not a geographic location, so it is not classified as a location.",no,"The phrase 'hooldeprojekti' was classified as 'no' because it does not indicate a location or answer the question 'where', but rather refers to a project related to care."
3080,27858061,eelnevasse,eelnev,ütlema,NaN,ill,18375902,Mind lihtsalt huvitas kuidas Sa vastaksid .. neile kahele küsimusele ja ka sellele kolmandale mille Teekäija esitas .. tõesti sekkumata üldse eelnevasse kus mida keegi ütles - tahtsin teada lihtsalt mida Sina arvad v ütled selle kohta !,NaN,NaN,NaN,no,"The word 'eelnevasse' refers to a preceding context or matter, not a physical or geographical location.",no,"The phrase 'eelnevasse' refers to something previously mentioned or an earlier situation, not indicating a specific place."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6144,5368550,Lennuplaani,lennuplaan,liikuma,NaN,adit,3338090,"Lennuplaani järgselt Kaliningradi suunas liikunud lennuk sisenes Eesti õhuruumi Vaindloo saare piirkonnas ühe meremiili sügavuselt , viibides Eesti õhuruumis alla minuti .",NaN,NaN,NaN,no,The phrase 'Lennuplaani' refers to a flight schedule and not a physical location.,no,"The phrase 'Lennuplaani' refers to a flight schedule, which is a temporal or organizational concept rather than a physical location or place, so it is not adverbial of place."
8809,2280193,14-s,14,käima,NaN,in,1433193,"110-st valimisringkonnast 14-s , nende seas ka kolmes pealinna Minski ringkonnas , ei käinud komisjoni väitel oma häält andmas üle poole valijatest ja seal korraldatakse valimiste teine voor .",NaN,NaN,NaN,no,"The phrase '14-s' refers to an ordinal number and not a physical location, hence it was classified as 'no'.",no,"The phrase '14-s' refers to the specific number of constituencies but does not describe a physical location or place, so it is not adverbial of place."
4240,17972665,